## Many well

## Ground truth

In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np

# ---------------------------
# 1. Parameters
# ---------------------------
d = 5
delta = 4.0
N_GRID = 20001
N_SAMPLES = 4000

# ---------------------------
# 2. Compute normalized 1D PDF (host-side)
# ---------------------------
x_grid = jnp.linspace(-5.0, 5.0, N_GRID)
U_1d = (x_grid**2 - delta) ** 2
unnorm = jnp.exp(-U_1d)

# use modern numpy trapezoid for integration
Z_1d = np.trapezoid(np.array(unnorm), np.array(x_grid))
pdf_1d = unnorm / Z_1d
cdf_1d = jnp.cumsum(pdf_1d)
cdf_1d = cdf_1d / cdf_1d[-1]

# ---------------------------
# 3. Inverse-CDF sampler (no JIT here due to dynamic n)
# ---------------------------
def sample_1d(key, n):
    u = random.uniform(key, shape=(n,))
    idx = jnp.searchsorted(cdf_1d, u, side="right")
    idx = jnp.clip(idx, 1, len(x_grid) - 1)
    x0, x1 = x_grid[idx - 1], x_grid[idx]
    c0, c1 = cdf_1d[idx - 1], cdf_1d[idx]
    slope = (x1 - x0) / (c1 - c0)
    return x0 + slope * (u - c0)

# ---------------------------
# 4. Generate 5D samples (coordinates independent)
# ---------------------------
def sample_5d(key, n):
    subkeys = random.split(key, d)
    samples = [sample_1d(k, n) for k in subkeys]
    return jnp.stack(samples, axis=1)

# ---------------------------
# 5. Draw samples
# ---------------------------
rng = random.PRNGKey(42)
samples_5d = sample_5d(rng, N_SAMPLES)
ref_samples_5d   = jnp.array(samples_5d)

print("5D samples shape:", samples_5d.shape)
print("Example samples:\n", np.array(samples_5d[:5]))


5D samples shape: (4000, 5)
Example samples:
 [[ 1.6810433  1.9631404  1.9049636 -1.8410207  1.9777675]
 [-1.9249195  2.0174553  1.9574592 -2.1475084  1.7762471]
 [ 2.1310716 -2.0457478 -2.0996935 -2.200176   2.1933317]
 [ 1.9357337 -1.9724945 -1.8825396  1.8541056  2.0106044]
 [ 1.8495694 -2.117268  -1.7942214 -1.8868984  2.004968 ]]


## PPO Run

In [ ]:
# =========================================================
# PPO-RLFS for Many-Well (MW54) with Adaptive Annealing — Fixed
# =========================================================
import jax
import jax.numpy as jnp
from jax import random, vmap, lax, jit
import optax
from flax import linen as nn
from flax.training.train_state import TrainState
import numpy as np

# =========================================================
# 1. Target: Independent Double-Well Potential
# =========================================================
D = 5
DELTA = 4.0

def energy_doublewell(x, delta=DELTA):
    return jnp.sum((x**2 - delta)**2, axis=-1)

def log_unnormalized_pi(x):
    def single(xx): return -energy_doublewell(xx)
    return vmap(single)(x) if x.ndim == 2 else single(x)

def terminal_reward(x_T):
    return log_unnormalized_pi(x_T)

# =========================================================
# 2. RLFS transition
# =========================================================
SIGMA_INIT = 0.8
SIGMA_FINAL = 0.1
ENTROPY_COEF = 0.01
T_HORIZON = 32

def env_step(key, x_t, t_frac, mu_t, sigma, sigma_ref):
    """Stable forward/backward KL reward step."""
    B, dim = x_t.shape
    key_a, _ = random.split(key)
    eps = random.normal(key_a, (B, dim))
    a_t = mu_t + sigma * eps
    x_next = x_t + a_t        # <== smaller integration step for stability

    mean_f = x_t + mu_t
    diff_f = x_next - mean_f
    sigref2 = sigma**2
    logF = -0.5 * jnp.sum((diff_f**2) / sigref2, axis=1)

    coef = jnp.sqrt(1.0 - sigma**2)
    mean_b = coef * x_next
    diff_b = x_t - mean_b
    logB = -0.5 * jnp.sum((diff_b**2) / sigref2, axis=1)

    r_t = logB - logF
    diff_a = (a_t - mu_t) / sigma
    logp = -0.5 * jnp.sum(diff_a**2, axis=1)
    return x_next, a_t, logp, r_t

# =========================================================
# Actor / Critic Networks (bounded)
# =========================================================
class Actor(nn.Module):
    D: int
    hidden: int = 256
    @nn.compact
    def __call__(self, x, t_frac):
        h = jnp.concatenate([x, t_frac], axis=-1)
        h = nn.tanh(nn.Dense(self.hidden)(h))
        h = nn.tanh(nn.Dense(self.hidden)(h))
        mu = nn.Dense(self.D)(h)
        return mu

class Critic(nn.Module):
    hidden: int = 256
    @nn.compact
    def __call__(self, x, t_frac):
        h = jnp.concatenate([x, t_frac], axis=-1)
        h = nn.tanh(nn.Dense(self.hidden)(h))
        h = nn.tanh(nn.Dense(self.hidden)(h))
        v = nn.Dense(1)(h)
        return v.squeeze(-1)

class ActorState(TrainState): sigma: float
class CriticState(TrainState): pass

# =========================================================
# 3. MMD metric (robust)
# =========================================================
def pairwise_sq_dists(X, Y):
    X_norm = jnp.sum(X**2, axis=1, keepdims=True)
    Y_norm = jnp.sum(Y**2, axis=1, keepdims=True).T
    return X_norm + Y_norm - 2.0 * (X @ Y.T)

def rbf_kernel_matrix(X, Y, gamma):
    d2 = pairwise_sq_dists(X, Y)
    return jnp.exp(-gamma * d2)

def median_kernel_width(X, Y):
    Z = jnp.concatenate([X, Y], axis=0)
    d2 = pairwise_sq_dists(Z, Z)
    d2_flat = d2[jnp.triu_indices(d2.shape[0], k=1)]
    d2_flat = d2_flat[jnp.isfinite(d2_flat)]
    if d2_flat.size == 0:
        return 1.0
    med = jnp.median(jnp.sqrt(jnp.clip(d2_flat, 1e-6, 1e6)))
    med = jnp.maximum(med, 1e-3)
    sigma2 = med**2
    return 1.0 / (2.0 * sigma2)

def mmd_unbiased_rbf(X, Y):
    X = X[jnp.isfinite(X).all(axis=1)]
    Y = Y[jnp.isfinite(Y).all(axis=1)]
    n, m = X.shape[0], Y.shape[0]
    if n < 2 or m < 2:
        return jnp.nan
    gamma = median_kernel_width(X, Y)
    Kxx = rbf_kernel_matrix(X, X, gamma)
    Kyy = rbf_kernel_matrix(Y, Y, gamma)
    Kxy = rbf_kernel_matrix(X, Y, gamma)
    sum_Kxx = (jnp.sum(Kxx) - jnp.trace(Kxx)) / (n * (n - 1) + 1e-12)
    sum_Kyy = (jnp.sum(Kyy) - jnp.trace(Kyy)) / (m * (m - 1) + 1e-12)
    sum_Kxy = jnp.sum(Kxy) / (n * m + 1e-12)
    val = sum_Kxx + sum_Kyy - 2.0 * sum_Kxy
    return jnp.nan_to_num(val, nan=0.0, posinf=0.0, neginf=0.0)

def mode_sign_pattern(x):
    bits = (x >= 0).astype(jnp.int32)
    powers = (2 ** jnp.arange(x.shape[1], dtype=jnp.int32))
    codes = jnp.sum(bits * powers[None, :], axis=1)
    return codes

def mode_coverage_stats(x, D):
    codes = mode_sign_pattern(x)
    unique_codes, counts = jnp.unique(codes, return_counts=True)
    n_modes_hit = unique_codes.shape[0]
    full_K = 2 ** D
    hist = jnp.zeros((full_K,), dtype=jnp.int32).at[unique_codes].set(counts)
    freq = hist / jnp.sum(hist)
    return n_modes_hit, freq

# =========================================================
# 4. Rollout + GAE
# =========================================================
def rollout_episode(rng, actor_params, critic_params,
                    actor_apply, critic_apply,
                    BATCH, D, T, sigma, sigma_ref):
    rng, key_init = random.split(rng)
    x0 = 2.0 * random.normal(key_init, (BATCH, D))

    def body(carry, t_idx):
        rng, x_t = carry
        t_frac = jnp.ones((BATCH, 1)) * (t_idx / T)
        mu_t = actor_apply(actor_params, x_t, t_frac)
        rng, step_key = random.split(rng)
        x_next, a_t, logp, r_t = env_step(step_key, x_t, t_frac, mu_t, sigma, sigma_ref)
        v_t = critic_apply(critic_params, x_t, t_frac)
        return (rng, x_next), {
            "x_t": x_t, "a_t": a_t, "logprob": logp,
            "value": v_t, "t_frac": t_frac.squeeze(-1),
            "rewards": r_t
        }

    (rng, x_T), traj = lax.scan(body, (rng, x0), jnp.arange(T))

    # Compute terminal reward and make final step terminal (no bootstrapping)
    rew_T = terminal_reward(x_T)
    rewards = jnp.stack(traj["rewards"], axis=0)
    rewards = rewards.at[-1].add(rew_T)

    v_T = jnp.zeros((BATCH,))  # <-- FIXED: true terminal, no future bootstrap

    traj["rewards"] = rewards
    traj["v_T"] = v_T
    traj["x_T"] = x_T
    traj["terminal_bonus"] = rew_T
    return rng, traj


def compute_gae(traj, gamma=1.0, lam=0.95):
    rewards, values, v_T = traj["rewards"], traj["value"], traj["v_T"]
    T_len, B = rewards.shape
    def backward(carry, t_idx):
        gae, next_value = carry
        r_t, v_t = rewards[t_idx], values[t_idx]
        delta = r_t + gamma * next_value - v_t
        gae_t = delta + gamma * lam * gae
        return (gae_t, v_t), gae_t
    (_, _), adv_rev = lax.scan(backward, (jnp.zeros((B,)), v_T), jnp.arange(T_len-1, -1, -1))
    advantages = jnp.flip(adv_rev, axis=0)
    returns = advantages + values
    return advantages, returns

# =========================================================
# 5. PPO loss
# =========================================================
def gaussian_logprob(mu, sigma, a):
    diff = (a - mu) / sigma
    return -0.5 * jnp.sum(diff**2, axis=-1)

def ppo_loss(actor_params, critic_params, batch, actor_apply, critic_apply,
             sigma, clip_eps=0.2, vf_coef=0.5, entropy_coef=ENTROPY_COEF):
    x_t, t_frac, a_t, old_lp, adv_t, ret_t = (
        batch["x_t"], batch["t_frac"], batch["a_t"],
        batch["logprob"], batch["adv"], batch["ret"]
    )
    mu = actor_apply(actor_params, x_t, t_frac[:,None])
    new_lp = gaussian_logprob(mu, sigma, a_t)
    ratio = jnp.exp(new_lp - old_lp)
    adv_norm = (adv_t - jnp.mean(adv_t)) / (jnp.std(adv_t) + 1e-8)
    surr1 = ratio * adv_norm
    surr2 = jnp.clip(ratio, 1.0-clip_eps, 1.0+clip_eps) * adv_norm
    actor_obj = -jnp.mean(jnp.minimum(surr1, surr2))
    v_pred = critic_apply(critic_params, x_t, t_frac[:,None])
    v_loss = jnp.mean((ret_t - v_pred)**2)
    dim = a_t.shape[-1]
    entropy = 0.5 * jnp.mean(dim * (1.0 + jnp.log(2*jnp.pi*(sigma**2))))
    total_loss = actor_obj + vf_coef*v_loss - entropy_coef*entropy
    aux = {"actor_loss": actor_obj, "critic_loss": v_loss,
           "ratio_mean": jnp.mean(ratio), "entropy": entropy}
    return total_loss, aux

# =========================================================
# 6. Annealing Schedules
# =========================================================
def sigma_schedule(it, sigma_init=SIGMA_INIT, sigma_final=SIGMA_FINAL, freeze_iter=3000):
    frac = jnp.clip(it / freeze_iter, 0.0, 1.0)
    return sigma_init + frac * (sigma_final - sigma_init)

def entropy_coef_schedule(it, warmup_iter=500, base_coef=0.01, boosted_coef=0.05):
    return jnp.where(it < warmup_iter, base_coef, boosted_coef)

# =========================================================
# 7. Training
# =========================================================
def train_manywell_ppo(rng_key=random.PRNGKey(0), D=5, BATCH=1024, T=T_HORIZON,
                       actor_lr=1e-5, critic_lr=1e-5, ppo_epochs=3,
                       clip_eps=0.2, NITER=1000, MINIBATCH=20000):
    actor, critic = Actor(D=D, hidden=256), Critic(hidden=256)
    rng_key, key_actor, key_critic = random.split(rng_key, 3)
    dummy_x, dummy_t = random.normal(key_actor, (1,D)), jnp.zeros((1,1))
    actor_params, critic_params = actor.init(key_actor, dummy_x, dummy_t), critic.init(key_critic, dummy_x, dummy_t)
    # gradient clipping
    actor_opt  = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(actor_lr))
    critic_opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(critic_lr))
    actor_state = ActorState.create(apply_fn=actor.apply, params=actor_params, tx=actor_opt, sigma=SIGMA_INIT)
    critic_state = CriticState.create(apply_fn=critic.apply, params=critic_params, tx=critic_opt)
    ref_samples_5d = 2.0 * (2 * (random.randint(rng_key, (2000, D), 0, 2) - 0.5))

    @jit
    def do_rollout(rng_key, actor_params, critic_params, sigma, sigma_ref):
        rng_key, traj = rollout_episode(rng_key, actor_params, critic_params,
                                        actor.apply, critic.apply,
                                        BATCH, D, T, sigma, sigma_ref)
        adv, ret = compute_gae(traj)
        batch = {
            "x_t": traj["x_t"].reshape(T * BATCH, D),
            "a_t": traj["a_t"].reshape(T * BATCH, D),
            "t_frac": traj["t_frac"].reshape(T * BATCH),
            "logprob": traj["logprob"].reshape(T * BATCH),
            "adv": adv.reshape(T * BATCH),
            "ret": ret.reshape(T * BATCH),
        }
        mean_logpiT = jnp.mean(traj["terminal_bonus"])
        logpi_T = traj["terminal_bonus"]
        return rng_key, batch, mean_logpiT, traj["x_T"], logpi_T

    @jit
    def ppo_update(actor_state, critic_state, batch, sigma, ent_coef):
        def loss_fn_actor(params):
            loss_val, aux = ppo_loss(params, critic_state.params, batch,
                                     actor_state.apply_fn, critic_state.apply_fn,
                                     sigma=sigma, clip_eps=clip_eps, entropy_coef=ent_coef)
            return loss_val, aux
        def critic_only_loss(params):
            _, aux = ppo_loss(actor_state.params, params, batch,
                              actor_state.apply_fn, critic_state.apply_fn,
                              sigma=sigma, clip_eps=clip_eps, entropy_coef=ent_coef)
            return aux["critic_loss"], aux
        actor_grads, actor_aux = jax.grad(loss_fn_actor, has_aux=True)(actor_state.params)
        critic_grads, critic_aux = jax.grad(critic_only_loss, has_aux=True)(critic_state.params)
        new_actor_state = actor_state.apply_gradients(grads=actor_grads)
        new_critic_state = critic_state.apply_gradients(grads=critic_grads)
        aux = {"actor_loss": actor_aux["actor_loss"],
               "critic_loss": critic_aux["critic_loss"],
               "ratio_mean": actor_aux["ratio_mean"],
               "entropy": actor_aux["entropy"]}
        return new_actor_state, new_critic_state, aux

    for it in range(1, NITER+1):
        sigma_now = float(sigma_schedule(it))
        ent_coef  = float(entropy_coef_schedule(it))
        rng_key, batch, mean_logpiT, x_T, logpi_T = do_rollout(
            rng_key, actor_state.params, critic_state.params, sigma_now, sigma_now
        )

        N_total = batch["x_t"].shape[0]
        idx = jnp.arange(N_total)
        for _ in range(ppo_epochs):
            rng_key, subkey = random.split(rng_key)
            perm = random.permutation(subkey, idx)
            for start in range(0, N_total, MINIBATCH):
                mb_idx = perm[start:start+MINIBATCH]
                minibatch = {k: v[mb_idx] for k, v in batch.items()}
                actor_state, critic_state, aux = ppo_update(actor_state.replace(sigma=sigma_now),
                                                            critic_state, minibatch,
                                                            sigma_now, ent_coef)
        if it % 100 == 0:
            q1, q2, q3 = jnp.percentile(logpi_T, jnp.array([25,50,75]))
            iqr = q3 - q1
            print(f"[Iter {it:04d}] σ={sigma_now:.3f} log ν(x_T) quartiles=({q1:.1f},{q2:.1f},{q3:.1f}) "
                  f"IQR={iqr:.1f} | actor_loss={aux['actor_loss']:.3f} "
                  f"critic_loss={aux['critic_loss']:.3f} ratio={aux['ratio_mean']:.3f} H={aux['entropy']:.3f}")
            model_eval, ref_eval = x_T[:1000], ref_samples_5d[:1000]
            mmd2_val = float(mmd_unbiased_rbf(model_eval, ref_eval))
            n_modes, freq = mode_coverage_stats(x_T, D)
            top, bottom = jnp.sort(freq)[-1], jnp.sort(freq)[0]
            print(f"   ↳ modes_hit={int(n_modes)}/{2**D} | top mode mass={float(top):.3f} | bottom mode mass={float(bottom):.3f}")
            print(f"   ↳ MMD²={mmd2_val:.4e}")

    return actor_state, critic_state

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    actor_state, critic_state = train_manywell_ppo(
        rng_key=random.PRNGKey(0),
        D=D,
        BATCH=1024,
        T=50,
        actor_lr=1e-4,
        critic_lr=1e-4,
        ppo_epochs=4,
        clip_eps=0.2,
        NITER=4000,
        MINIBATCH=5000
    )

[Iter 0100] σ=0.777 log ν(x_T) quartiles=(-58.0,-49.7,-41.3) IQR=16.7 | actor_loss=0.000 critic_loss=88.757 ratio=1.001 H=5.831
   ↳ modes_hit=32/32 | top mode mass=0.063 | bottom mode mass=0.006
   ↳ MMD²=5.6965e-02
[Iter 0200] σ=0.753 log ν(x_T) quartiles=(-56.0,-46.3,-37.4) IQR=18.7 | actor_loss=-0.004 critic_loss=109.700 ratio=1.000 H=5.678
   ↳ modes_hit=32/32 | top mode mass=0.084 | bottom mode mass=0.006
   ↳ MMD²=4.8332e-02
[Iter 0300] σ=0.730 log ν(x_T) quartiles=(-54.4,-44.5,-36.1) IQR=18.3 | actor_loss=-0.000 critic_loss=68.207 ratio=0.999 H=5.521
   ↳ modes_hit=32/32 | top mode mass=0.126 | bottom mode mass=0.007
   ↳ MMD²=5.4476e-02
[Iter 0400] σ=0.707 log ν(x_T) quartiles=(-51.2,-40.8,-32.1) IQR=19.2 | actor_loss=-0.006 critic_loss=117.672 ratio=1.001 H=5.359
   ↳ modes_hit=32/32 | top mode mass=0.225 | bottom mode mass=0.001
   ↳ MMD²=9.1873e-02
[Iter 0500] σ=0.683 log ν(x_T) quartiles=(-46.3,-36.7,-26.7) IQR=19.7 | actor_loss=-0.004 critic_loss=108.164 ratio=0.999 H=5.1

In [32]:
import jax
import jax.numpy as jnp
from jax import lax, random
from jax.scipy.special import logsumexp

# =========================================================
# Pairwise squared distances
# =========================================================
def pairwise_sq_dists(X, Y):
    X2 = jnp.sum(X**2, axis=1, keepdims=True)
    Y2 = jnp.sum(Y**2, axis=1, keepdims=True).T
    return X2 + Y2 - 2.0 * (X @ Y.T)

# =========================================================
# Stable Sinkhorn divergence (Genevay et al., 2019)
# =========================================================
def _sinkhorn_cost_logdomain(C, a, b, eps, n_iters):
    """Compute entropic OT cost W_eps(a,b) in a stable log-domain form."""
    log_a = jnp.log(a)
    log_b = jnp.log(b)
    f = jnp.zeros_like(a)
    g = jnp.zeros_like(b)

    def body_fn(_, state):
        f, g = state
        term_f = (g[None, :] - C) / eps + log_b[None, :]
        f = eps * (log_a - logsumexp(term_f, axis=1))
        term_g = (f[:, None] - C) / eps + log_a[:, None]
        g = eps * (log_b - logsumexp(term_g, axis=0))
        return (f, g)

    f, g = lax.fori_loop(0, n_iters, body_fn, (f, g))
    return jnp.sum(a * f) + jnp.sum(b * g)

def sinkhorn_divergence(X, Y, eps=1e-2, n_iters=100):
    """Compute stabilized Sinkhorn divergence S_eps(X,Y)."""
    X = X[jnp.isfinite(X).all(axis=1)]
    Y = Y[jnp.isfinite(Y).all(axis=1)]

    N, M = X.shape[0], Y.shape[0]
    a = jnp.full((N,), 1.0 / N)
    b = jnp.full((M,), 1.0 / M)

    C_xy = pairwise_sq_dists(X, Y)
    C_xx = pairwise_sq_dists(X, X)
    C_yy = pairwise_sq_dists(Y, Y)

    W_xy = _sinkhorn_cost_logdomain(C_xy, a, b, eps, n_iters)
    W_xx = _sinkhorn_cost_logdomain(C_xx, a, a, eps, n_iters)
    W_yy = _sinkhorn_cost_logdomain(C_yy, b, b, eps, n_iters)

    return W_xy - 0.5 * (W_xx + W_yy)

# =========================================================
# Sampling from trained PPO-RLFS actor
# =========================================================
def generate_samples(rng_key, actor_state, D=5, T=50, N=2000):
    """Generate final samples x_T from trained actor."""
    actor_apply = actor_state.apply_fn
    params = actor_state.params
    sigma = actor_state.sigma

    rng_key, key_init = random.split(rng_key)
    x0 = 2.0 * random.normal(key_init, (N, D))

    def body(carry, t_idx):
        rng, x_t = carry
        t_frac = jnp.ones((N, 1)) * (t_idx / T)
        mu_t = actor_apply(params, x_t, t_frac)
        rng, step_key = random.split(rng)
        eps = random.normal(step_key, (N, D))
        a_t = mu_t + sigma * eps
        x_next = x_t + a_t
        return (rng, x_next), x_next

    (_, x_T_seq) = lax.scan(body, (rng_key, x0), jnp.arange(T))
    x_T = x_T_seq[-1]
    return x_T

# =========================================================
# Standardization for evaluation
# =========================================================
def standardize_for_eval(X, Y):
    """Use reference statistics to standardize both samples."""
    mean = jnp.mean(Y, axis=0, keepdims=True)
    std = jnp.std(Y, axis=0, keepdims=True) + 1e-8
    return (X - mean) / std, (Y - mean) / std

# =========================================================
# Evaluation
# =========================================================
test_samples_5d = jnp.array(generate_samples(rng, actor_state, D=5, T=50, N=2000))
X_eval, Y_eval = standardize_for_eval(test_samples_5d, ref_samples_5d)
sd = sinkhorn_divergence(X_eval, Y_eval, eps=1e-2, n_iters=100)
mmd = mmd_unbiased_rbf(X_eval, Y_eval)
print("MMD²:", mmd)
print("Sinkhorn divergence:", sd)


MMD²: 0.0009794235
Sinkhorn divergence: 0.23727006
